# Score & rank jobs against the résumé

Load the eval jsonl (from `scrape.ipynb`'s eval step), embed each JD on the
GPU, compare against the résumé embeddings (from `resume.ipynb`), and rank
every job into `out/ranked_jobs.csv`. All logic lives in `src/score.py`.

Prerequisites: run `resume.ipynb` (chunk + embed) and `scrape.ipynb`
(scrape → revise → eval) first, so both inputs exist.


## Setup: make the project's `src/` importable


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from score import (
    DATA_DIR,
    OUT_CSV,
    aligned_resume_bullets,
    latest_eval,
    load_jobs,
    load_or_embed,
    rank,
    write_csv,
)

print(f"Ranked CSV -> {OUT_CSV.relative_to(ROOT)}")

Ranked CSV -> out/ranked_jobs.csv


## Load the inputs

Résumé embeddings (cache hit if unchanged), aligned to their bullets, and the
newest `*_eval.jsonl` validated into typed `Job` models.


In [2]:
ids, resume_vecs = load_or_embed()
resume_bullets = aligned_resume_bullets(ids)

jobs = load_jobs(latest_eval(DATA_DIR))

Cache hit — 26 embeddings from cache/resume_embeddings.npz
Loaded: 40 jobs
Skipped: 0
Labels seen: responsibility=447, requirement=379, preferred=47, context=341


## Rank

Embeds each JD on the GPU (cached per job under `cache/jd_embeddings/`), scores
it against the résumé, and writes the sorted CSV. Re-runs are a cache hit.


In [3]:
rows = rank(jobs, resume_vecs, resume_bullets)
write_csv(rows, OUT_CSV)
print(f"Wrote {len(rows)} ranked jobs -> {OUT_CSV.relative_to(ROOT)}")

ranking: 100%|██████████| 40/40 [00:00<00:00, 496.68job/s]

Embedded 0 job(s), 40 from cache
Wrote 40 ranked jobs -> out/ranked_jobs.csv


## Top matches

Composite score plus the per-label breakdown, best first.


In [4]:
import pandas as pd

pd.DataFrame(rows)[
    ["rank", "composite", "company", "position",
     "requirement", "responsibility", "preferred", "context"]
].head(15)

,rank,composite,company,position,requirement,responsibility,preferred,context
0,1,0.6909,Asana,"Senior Engineering Manager, AI Agents",0.7137,0.6787,0.6747,0.5842
1,2,0.6808,Asana,"Senior Engineering Manager, AI Agents",0.6989,0.6695,,0.5683
2,3,0.6805,Jobgether,"Engineering Manager, Data Platform & ML Ops",0.6972,0.6663,,0.5987
3,4,0.6799,PheedLoop,Senior Software (Engineering Manager Developme...,0.6930,0.6782,,0.5597
4,5,0.6760,Guardian Life,Head of MarCom Technology,0.6775,0.6829,0.6838,0.5973
5,6,0.6681,Movable Ink,"Manager, Engineering",0.6875,0.6482,,0.5942
6,7,0.6665,Roku,"Engineering Manager, Ad Serving",0.7104,0.6611,0.5796,0.5208
7,8,0.6661,Jobgether,Engineering Manager,0.6827,0.6500,0.6627,0.6061
8,9,0.6577,Robinhood,"Engineering Manager, Toronto",0.6684,0.6759,0.6292,0.5272
9,10,0.6554,McDonough Manufacturing Company,Engineering Manager,0.6531,0.6721,,0.5778
